In [3]:
import pandas as pd
import os
import re
import html
import ast
import sys
from bs4 import BeautifulSoup, NavigableString
from collections import deque, defaultdict
from tqdm import tqdm
SAMPLE_SIZE = 1000

KEYWORD_FILE_PATH = '/Users/wanghao/Downloads/keywords.xlsx' 

try:
    if KEYWORD_FILE_PATH.endswith('.xlsx'):
        df_kw_config = pd.read_excel(KEYWORD_FILE_PATH)
    else:
        df_kw_config = pd.read_csv(KEYWORD_FILE_PATH)

    KEYWORD_RULES = {}
    for _, row in df_kw_config.iterrows():
        k = str(row['Keyword']).strip()
        t = str(row['Search_Type']).strip().lower()
        KEYWORD_RULES[k] = t
    print(f"Loaded {len(KEYWORD_RULES)} keywords.")
except Exception as e:
    print(f"Error loading keywords: {e}")
    KEYWORD_RULES = {}

Loaded 3 keywords.


### 合併有關鍵字的檔案

In [7]:
all_excel_paths = [
    '/Volumes/One Touch/8k_has_keyword_file_html/summary/summary_1_to_200000.xlsx',
    '/Volumes/One Touch/8k_has_keyword_file_html/summary/summary_200000_to_400000.xlsx',
    '/Volumes/One Touch/8k_has_keyword_file_html/summary/summary_400000_to_600000.xlsx',
    '/Volumes/One Touch/8k_has_keyword_file_html/summary/summary_600000_to_690853.xlsx',
    '/Volumes/One Touch/8k_has_keyword_file_txt/summary_1_to_136955.xlsx'
]


# 讀取所有 Excel 檔，並把它們放進一個列表
list_of_dfs = [pd.read_excel(f) for f in all_excel_paths]

# 合併所有 DataFrame
df_merged_summaries = pd.concat(list_of_dfs, ignore_index=True)

print("\n--- 合併成功！ ---")
print(f"總筆數: {len(df_merged_summaries)}")
display(df_merged_summaries.head())


--- 合併成功！ ---
總筆數: 114511


,file_name,total_keyword_occurrences,num_distinct_keywords,keywords_found
0,MOTIVE_INC_0001193125-07-165389.html,91,1,['audit']
1,MOTIVE_INC_0001193125-07-183938.html,86,1,['audit']
2,FIRST_INDUSTRIAL_REALTY_TRUST_INC_0000950137-0...,60,1,['audit']
3,FIRST_INDUSTRIAL_LP_0000950137-07-006358.html,54,1,['audit']
4,HNET_NET_0001174239-02-000006.html,51,2,"['audit', 'auditor']"


# 

In [9]:
# --- 假設 df_merged_summaries 已經在您的 Notebook 記憶體中 ---
# (如果您需要重新整理，請執行上一段合併 Excel 的程式碼)

print("--- 正在解碼 file_name ---")

# 1. 定義 Regex
# (.*?)     -> 第 1 組 (coname): 非貪婪地抓取所有東西
# _         -> 匹配最後一個底線
# (\d{10}-\d{2}-\d{6}) -> 第 2 組 (accession): 精確抓取 Accession 號碼
# \..* -> 匹配副檔名 (例如 .html)
FILENAME_PATTERN = r"^(.*?)_(\d{10}-\d{2}-\d{6})\..*$"

# 2. 使用 .str.extract() 一次性抓取 (最簡單的語法)
# 這會自動建立 'coname_decoded' 和 'accession' 兩個新欄位
df_merged_summaries[['coname_decoded', 'accession']] = df_merged_summaries['file_name'].str.extract(FILENAME_PATTERN)

# 3. 建立我們最終的「工作清單」 (移除解碼失敗的)
df_jobs = df_merged_summaries.dropna(subset=['accession'])

print(f"解碼完成。成功解碼 {len(df_jobs)} 筆。")
print("\n--- 解碼後的「工作清單」(df_jobs) 預覽 (前 5 筆)： ---")
display(df_jobs.head())

--- 正在解碼 file_name ---
解碼完成。成功解碼 114511 筆。

--- 解碼後的「工作清單」(df_jobs) 預覽 (前 5 筆)： ---


,file_name,total_keyword_occurrences,num_distinct_keywords,keywords_found,coname_decoded,accession
0,MOTIVE_INC_0001193125-07-165389.html,91,1,['audit'],MOTIVE_INC,0001193125-07-165389
1,MOTIVE_INC_0001193125-07-183938.html,86,1,['audit'],MOTIVE_INC,0001193125-07-183938
2,FIRST_INDUSTRIAL_REALTY_TRUST_INC_0000950137-0...,60,1,['audit'],FIRST_INDUSTRIAL_REALTY_TRUST_INC,0000950137-07-006354
3,FIRST_INDUSTRIAL_LP_0000950137-07-006358.html,54,1,['audit'],FIRST_INDUSTRIAL_LP,0000950137-07-006358
4,HNET_NET_0001174239-02-000006.html,51,2,"['audit', 'auditor']",HNET_NET,0001174239-02-000006


### 在8K_Filings找出有關鍵字的檔案並把他們的欄位和df_jobs結合

In [11]:
# --- 你的函式 (不變) ---
def _sanitize_filename(name: str) -> str:
    """
    清理 coname，移除非法字元，並將空白換成底線。
    """
    name = str(name)
    name = re.sub(r'[\\/*?:"<>|]', "", name)
    name = name.replace(" ", "_")
    return name.strip()

# --- 1. 讀取與準備資料 ---
try:
    # 讀取 Excel，並確保 coname 和 accession 被視為字串
    df_8k = pd.read_excel(
        '/Users/wanghao/8k/8K_Filings.xlsx', 
        dtype={'coname': str, 'accession': str, 'cik': str}
    )
except FileNotFoundError:
    print("錯誤：找不到 '8K_Filings.xlsx'。")
    # 根據你的圖片和新規則使用模擬資料
   
# 在處理前，先儲存原始欄位名稱
original_8k_cols = list(df_8k.columns)


print("--- 讀取的 8K Filings (df_8k) ---")
print(df_8k.head())

# --- 2. 處理 df_8k，建立 'base_name' (依照你的新規則) ---

# * 修正 *：只對 coname 執行 sanitize
df_8k['coname_safe'] = df_8k['coname'].apply(_sanitize_filename)

# * 修正 *：accession 保持原樣 (確保是字串)
df_8k['accession_raw'] = df_8k['accession'].astype(str)

# 組合成基準檔名： (sanitized_coname) + _ + (raw_accession)
df_8k['base_name'] = df_8k['coname_safe'] + '_' + df_8k['accession_raw']

print("\n--- 處理後的 df_8k (只清理 coname) ---")
print(df_8k[['coname', 'accession', 'base_name']].head())


# --- 3. 處理 df_summaries (邏輯不變：移除副檔名) ---

df_jobs['base_name'] = df_jobs['file_name'].str.rsplit('.', n=1).str[0]

print("\n--- 處理後的 df_summaries (移除副檔名) ---")
print(df_jobs)


# --- 4. 合併兩個 DataFrame ---

df_final_result = pd.merge(
    df_8k, 
    df_jobs, 
    on='base_name', 
    how='inner' # 'inner' 代表只保留雙方都有的
)

# --- 5. 擷取你需要的欄位 (使用 .drop() 移除不要的) ---

# 建立一個我們要「丟掉」的欄位清單
columns_to_drop = [
    # 1. 你明確指定不要的
    'fname', 
    
    # 2. 為了合併而產生的「暫時輔助欄位」
    'coname_safe',
    'accession_raw',
    'base_name',
    'coname_decoded', 
    'accession_y'
]

# 執行 drop，並使用 errors='ignore'
# 'ignore' 的好處是：就算上面清單的某個欄位不存在 (例如 'nitem' 其實是 'item')，
# 程式也不會報錯，會直接跳過。
final_selection = df_final_result.drop(
    columns=columns_to_drop,
    errors='ignore' 
)

final_selection_renamed = final_selection.rename(
    columns={
        'accession_x': 'accession'
    }
)
print("\n" + "="*30)
print("最終合併結果 ")
print("="*30)
print(final_selection_renamed.head())


--- 讀取的 8K Filings (df_8k) ---
       fdate                              coname         cik form  nitem  \
0 1994-01-04                   SOUTHERN UNION CO  0000203248  8-K   8.01   
1 1994-01-04  CLEVELAND ELECTRIC ILLUMINATING CO  0000020947  8-K   8.01   
2 1994-01-04                    TOLEDO EDISON CO  0000352049  8-K   8.01   
3 1994-01-04                    BROOKE GROUP LTD  0000059440  8-K   8.01   
4 1994-01-04                    MDC HOLDINGS INC  0000773141  8-K   8.01   

           item                                       fname  \
0  Other events  edgar/data/203248/0000913907-94-000002.txt   
1  Other events   edgar/data/20947/0000774197-94-000001.txt   
2  Other events  edgar/data/352049/0000774197-94-000001.txt   
3  Other events   edgar/data/59440/0000950123-94-000010.txt   
4  Other events  edgar/data/773141/0000950109-94-000014.txt   

              accession  
0  0000913907-94-000002  
1  0000774197-94-000001  
2  0000774197-94-000001  
3  0000950123-94-000010  
4  

### 利用剛剛做的final_selection 製作出html報表

In [19]:
SOURCE_DIRECTORIES = [
    '/Volumes/One Touch/8k_has_keyword_file_html/Processing_Files_1_to_200000',
    '/Volumes/One Touch/8k_has_keyword_file_html/Processing_Files_200000_to_400000',
    '/Volumes/One Touch/8k_has_keyword_file_html/Processing_Files_400000_to_600000',
    '/Volumes/One Touch/8k_has_keyword_file_html/Processing_Files_600000_to_690853',
    '/Volumes/One Touch/8k_has_keyword_file_txt/Processing_Files_1_to_136955'
]
OUTPUT_DIR = '/Volumes/One Touch/result'
BLOCK_TAGS = ['p', 'td', 'th', 'li', 'div', 'blockquote', 'pre']

PROCESSED_DIR = os.path.join(OUTPUT_DIR, "processed_files")
REPORT_FILE_PATH = os.path.join(OUTPUT_DIR, "validation_report_8k_files.html")
os.makedirs(PROCESSED_DIR, exist_ok=True)

def find_source_file_path(file_name: str):
    for dir_path in SOURCE_DIRECTORIES:
        for root, _, files in os.walk(dir_path):
            if file_name in files:
                return os.path.join(root, file_name)
    return None

def get_anchor_id(file_name: str, keyword: str, index: int) -> str:
    clean_name = re.sub(r'[^a-zA-Z0-9_-]', '_', file_name.split('.')[0])
    clean_keyword = re.sub(r'[^a-zA-Z0-9]', '', keyword)
    return f"{clean_name}_{clean_keyword}_{index}"

def format_fdate(fdate_obj):
    if isinstance(fdate_obj, str):
        return fdate_obj.split(' ')[0]
    try:
        return fdate_obj.strftime('%Y-%m-%d')
    except Exception:
        return str(fdate_obj).split(' ')[0]

def process_txt_file(full_path, keywords_list, file_name):
    try:
        with open(full_path, 'r', encoding='utf-8', errors='ignore') as f:
            full_text = f.read()
    except Exception:
        return [], ""

    paragraphs = re.split(r'\n\s*\n', full_text)
    found_matches_for_report = []
    processed_paragraphs_for_file = []
    id_counter = 0

    regex_parts = []
    all_defined_keywords = list(KEYWORD_RULES.keys())
    sorted_keywords = sorted(all_defined_keywords, key=len, reverse=True)
    
    for kw in sorted_keywords:
        mode = KEYWORD_RULES.get(kw, 'prefix')
        safe_kw = re.escape(kw).replace(r"\ ", r"\s+").replace(" ", r"\s+")
        
        if mode == 'prefix':
            pattern = fr"\b{safe_kw}\w*(?:'s)?"
        elif mode == 'exact':
            pattern = fr"\b{safe_kw}(?:'s)?\b"
        else:
            pattern = safe_kw
            
        regex_parts.append(pattern)

    try:
        combined_pattern = '|'.join(regex_parts)
        pattern = re.compile(combined_pattern, re.IGNORECASE)
    except re.error:
        return [], ""

    for paragraph in paragraphs:
        if not paragraph.strip():
            processed_paragraphs_for_file.append(html.escape(paragraph))
            continue

        paragraph_escaped = html.escape(paragraph)
        replacements = []

        try:
            matches = list(pattern.finditer(paragraph_escaped))
        except re.error:
            matches = []

        for m in matches:
            id_counter += 1
            matched_text = m.group(0)
            anchor_id = get_anchor_id(file_name, matched_text, id_counter)
            
            found_matches_for_report.append({
                'keyword': matched_text,
                'paragraph_text': paragraph,
                'anchor_id': anchor_id,
                'keyword_count': id_counter
            })
            span = f'<span id="{anchor_id}" class="keyword-highlight">{matched_text}</span>'
            replacements.append((m.start(), m.end(), span))
        
        replacements.sort(key=lambda x: x[0], reverse=True)
        for start, end, span in replacements:
            paragraph_escaped = paragraph_escaped[:start] + span + paragraph_escaped[end:]

        processed_paragraphs_for_file.append(paragraph_escaped)

    css = """<style>
        .keyword-highlight { background-color: #FFA500; color: black; font-weight: bold; padding: 1px 3px; border-radius: 3px; }
        pre { white-space: pre-wrap; font-family: monospace; }
    </style>"""
    
    meta_tag = '<meta charset="UTF-8">'
    final_processed_html = f"<html><head>{meta_tag}{css}</head><body><pre>{'<br><br>'.join(processed_paragraphs_for_file)}</pre></body></html>"
    return found_matches_for_report, final_processed_html

def process_html_file(full_path, keywords_list, file_name):
    try:
        with open(full_path, 'r', encoding='utf-8', errors='ignore') as f:
            full_text = f.read()
    except Exception:
        return [], ""

    try:
        soup = BeautifulSoup(full_text, 'lxml')
    except Exception:
        soup = BeautifulSoup(full_text, 'html.parser')

    for s in soup(['script', 'style']):
        s.decompose()

    found_matches_for_report = []
    id_counter = 0

    regex_parts = []
    all_defined_keywords = list(KEYWORD_RULES.keys())
    sorted_keywords = sorted(all_defined_keywords, key=len, reverse=True)
    
    for kw in sorted_keywords:
        mode = KEYWORD_RULES.get(kw, 'prefix')
        safe_kw = re.escape(kw).replace(r"\ ", r"\s+").replace(" ", r"\s+")
        
        if mode == 'prefix':
            pattern = fr"\b{safe_kw}\w*(?:'s)?"
        elif mode == 'exact':
            pattern = fr"\b{safe_kw}(?:'s)?\b"
        else:
            pattern = safe_kw
        regex_parts.append(pattern)

    try:
        combined_pattern = '|'.join(regex_parts)
        pattern = re.compile(combined_pattern, re.IGNORECASE)
    except re.error:
         return [], str(soup)

    all_text_nodes = soup.find_all(string=True)
    
    for node in all_text_nodes:
        if node.parent.name in ['script', 'style', 'head', 'title']:
            continue
        original_text = str(node)
        if not original_text.strip():
            continue
            
        try:
            matches = list(pattern.finditer(original_text))
        except re.error:
            continue
            
        if not matches:
            continue

        context_node = node
        while context_node.parent and context_node.parent.name not in BLOCK_TAGS and context_node.parent.name != 'body':
            context_node = context_node.parent
        
        if context_node.parent:
            text_for_report = context_node.parent.get_text(separator=' ', strip=True)
        else:
            text_for_report = original_text

        replacements = []
        for m in matches:
            id_counter += 1
            kw_match = m.group(0)
            aid = get_anchor_id(file_name, kw_match, id_counter)
            
            found_matches_for_report.append({
                'keyword': kw_match,
                'paragraph_text': text_for_report,
                'anchor_id': aid,
                'keyword_count': id_counter
            })
            replacements.append((m.start(), m.end(), f'<span id="{aid}" class="keyword-highlight">{kw_match}</span>'))
        
        replacements.sort(key=lambda x: x[0], reverse=True)
        new_html_content = original_text
        for start, end, span in replacements:
            new_html_content = new_html_content[:start] + span + new_html_content[end:]
        
        node.replace_with(BeautifulSoup(new_html_content, 'html.parser'))

    try:
        meta_tag = '<meta charset="UTF-8">'
        css = """<style>
            .keyword-highlight { background-color: #FFA500; color: black; font-weight: bold; padding: 1px 3px; border-radius: 3px; }
            pre { white-space: pre-wrap; font-family: monospace; }
        </style>"""
        
        head = soup.find('head')
        if not head:
            head = soup.new_tag('head')
            html_tag = soup.find('html')
            if html_tag:
                html_tag.insert(0, head)
            else:
                soup.insert(0, head)

        head.append(BeautifulSoup(meta_tag, 'html.parser'))
        head.append(BeautifulSoup(css, 'html.parser'))
    except Exception:
        pass

    return found_matches_for_report, str(soup)

HTML_TEMPLATE_HEADER = f"""
<!DOCTYPE html>
<html lang="zh-Hant">
<head>
    <meta charset="UTF-8">
    <title>驗證報告 (抽樣 {SAMPLE_SIZE} 個檔案)</title>
    <style>
        body {{ font-family: Arial, sans-serif; margin: 20px; }}
        h1 {{ color: #333; }}
        table {{ border-collapse: collapse; width: 100%; border: 1px solid #ccc; font-size: 11px; table-layout: fixed; }}
        th, td {{ border: 1px solid #ccc; padding: 6px; text-align: left; vertical-align: top; word-wrap: break-word; }}
        th {{ background-color: #f2f2f2; position: sticky; top: 0; z-index: 10; }}
        tr:nth-child(even) {{ background-color: #f9f9f9; }}
        .col-fdate {{ width: 6%; }}
        .col-coname {{ width: 14%; }}
        .col-cik {{ width: 6%; }}
        .col-form {{ width: 4%; }}
        .col-nitem {{ width: 4%; }}
        .col-item {{ width: 8%; }}
        .col-accession {{ width: 12%; }}
        .col-stats-2 {{ width: 5%; }}
        .col-keyword {{ width: 6%; }}
        .col-text {{ width: 35%; }}
        .snippet-highlight-link {{ background-color: #FFA500; color: black; font-weight: bold; text-decoration: none; padding: 1px 2px; border-radius: 3px; }}
        .snippet-highlight-link:hover {{ text-decoration: underline; }}
    </style>
</head>
<body>
    <h1>驗證報告</h1>
    <p>抽樣 <b>{SAMPLE_SIZE}</b> 個檔案，共找到 <b id="total_hits">...</b> 筆關鍵字。</p>
    <table>
        <thead>
            <tr>
                <th class="col-fdate">fdate</th>
                <th class="col-coname">coname</th>
                <th class="col-cik">cik</th>
                <th class="col-form">form</th>
                <th class="col-nitem">nitem</th>
                <th class="col-item">item</th>
                <th class="col-accession">accession</th>
                <th class="col-stats-2">Keyword_Number</th>
                <th class="col-keyword">Keyword</th>
                <th class="col-text">text (Captured Paragraph)</th>
            </tr>
        </thead>
        <tbody>
"""

HTML_TEMPLATE_FOOTER = """
        </tbody>
    </table>
    <script>
        document.getElementById('total_hits').innerText = document.querySelectorAll('tbody tr').length;
    </script>
</body>
</html>
"""

if __name__ == "__main__":
    print(f"--- 驗證腳本啟動 (V7 - Span Anchor Fix) ---")
    print(f"輸出資料夾: {OUTPUT_DIR}")
    print(f"抽樣數量: {SAMPLE_SIZE} 檔案")

    try:
        _ = final_selection_renamed.head()
        print("偵測到 'final_selection_renamed' DataFrame。")
    except NameError:
        print("!! 錯誤：找不到 'final_selection_renamed' DataFrame。!!")
        print("請確保此腳本在已定義 'final_selection_renamed' 的環境中執行。")
        sys.exit(1)

    df_sample = (final_selection_renamed.sample(n=SAMPLE_SIZE, random_state=42)
                 if len(final_selection_renamed) > SAMPLE_SIZE else final_selection_renamed)
    print(f"已隨機抽取 {len(df_sample)} 筆資料進行處理...")

    all_report_rows = []
    processed_file_cache = {}
    files_not_found_count = 0
    keyword_parse_error_count = 0
    total_matches_found = 0
    first_file_not_found = None
    first_parse_error_data = None

    pbar = tqdm(df_sample.itertuples(), total=len(df_sample), desc="處理中...")
    for row in pbar:
        file_name = row.file_name
        keywords_raw = row.keywords_found

        if isinstance(keywords_raw, str):
            try:
                keywords_list = ast.literal_eval(keywords_raw)
            except Exception:
                keyword_parse_error_count += 1
                if first_parse_error_data is None:
                    first_parse_error_data = keywords_raw
                continue
        elif isinstance(keywords_raw, list):
            keywords_list = keywords_raw
        else:
            keyword_parse_error_count += 1
            if first_parse_error_data is None:
                first_parse_error_data = str(keywords_raw)
            continue

        if not isinstance(keywords_list, list) or not keywords_list:
            continue

        source_path = find_source_file_path(file_name)
        if not source_path:
            files_not_found_count += 1
            if first_file_not_found is None:
                first_file_not_found = file_name
            continue

        file_type = 'txt' if file_name.lower().endswith('.txt') else 'html'

        if source_path not in processed_file_cache:
            try:
                if file_type == 'txt':
                    dest_file_name = file_name[:-4] + ".html"
                else:
                    dest_file_name = file_name
                dest_path = os.path.join(PROCESSED_DIR, dest_file_name)
                
                if file_type == 'txt':
                    matches_list, processed_html_content = process_txt_file(source_path, keywords_list, file_name)
                else:
                    matches_list, processed_html_content = process_html_file(source_path, keywords_list, file_name)

                with open(dest_path, 'w', encoding='utf-8') as f:
                    f.write(processed_html_content)

                processed_file_cache[source_path] = (dest_path, matches_list)
            except Exception:
                processed_file_cache[source_path] = (None, [])

        dest_path, matches_list = processed_file_cache[source_path]
        if not dest_path:
            continue

        total_matches_found += len(matches_list)
        pbar.set_description(
            f"處理中... (找到 {total_matches_found} 筆 | 檔案錯誤: {files_not_found_count} | 解析錯誤: {keyword_parse_error_count})"
        )

        keyword_in_para_counter = defaultdict(int)

        for match in matches_list:
            paragraph_text_raw = match['paragraph_text']
            keyword_text_raw = match['keyword']
            
            para_key = (paragraph_text_raw, keyword_text_raw.lower())
            
            keyword_in_para_counter[para_key] += 1
            n_th_occurrence = keyword_in_para_counter[para_key]

            paragraph_text = html.escape(paragraph_text_raw)
            keyword_text = html.escape(keyword_text_raw)
            fdate_str = format_fdate(getattr(row, 'fdate', ''))
            href = f"processed_files/{os.path.basename(dest_path)}#{match['anchor_id']}"
            replacement_link = f'<a href="{href}" target="_blank" class="snippet-highlight-link">{keyword_text}</a>'
            
            counter_list = [0]
            
            def callback_for_nth(match_obj):
                counter_list[0] += 1 
                if counter_list[0] == n_th_occurrence:
                    return replacement_link
                else:
                    return match_obj.group(0)

            try:
                pattern = r'\b' + re.escape(keyword_text) + r'\b'
                highlighted_paragraph = re.sub(
                    pattern,
                    callback_for_nth,
                    paragraph_text,
                    flags=re.IGNORECASE
                )
            except re.error:
                highlighted_paragraph = paragraph_text.replace(
                    keyword_text,
                    replacement_link
                )

            all_report_rows.append(f"""
            <tr>
                <td class="col-fdate">{fdate_str}</td>
                <td class="col-coname">{getattr(row, 'coname', '')}</td>
                <td class="col-cik">{getattr(row, 'cik', '')}</td>
                <td class="col-form">{getattr(row, 'form', '')}</td>
                <td class="col-nitem">{getattr(row, 'nitem', '')}</td>
                <td class="col-item">{getattr(row, 'item', '')}</td>
                <td class="col-accession">{getattr(row, 'accession', '')}</td> 
                <td class="col-stats-2">{match['keyword_count']}</td>
                <td class="col-keyword">{keyword_text}</td>
                <td class="col-text">{highlighted_paragraph}</td>
            </tr>
            """)

    pbar.close()
    print(f"\n處理完成，正在寫入總覽報告: {REPORT_FILE_PATH}")

    try:
        with open(REPORT_FILE_PATH, 'w', encoding='utf-8') as f:
            f.write(HTML_TEMPLATE_HEADER)
            f.write("\n".join(all_report_rows))
            f.write(HTML_TEMPLATE_FOOTER)
        print(f"--- 成功！ ---")
        print(f"{REPORT_FILE_PATH}")
    except Exception as e:
        print(f"--- 失敗！ ---")
        print(f"[嚴重錯誤] 無法寫入總覽報告: {e}")

    print("\n" + "="*30)
    print("V7 腳本診斷報告")
    print("="*30)
    print(f"總共處理 {len(df_sample)} 個檔案。")
    print(f"總共找到 {total_matches_found} 筆關鍵字實例。")
    print(f"找不到檔案: {files_not_found_count} 次")
    if first_file_not_found:
        print(f"  > 第一個找不到的檔案: '{first_file_not_found}'")
    print(f"關鍵字欄位解析錯誤: {keyword_parse_error_count} 次")
    if first_parse_error_data:
        print(f"  > 第一個解析失敗的資料: '{first_parse_error_data}'")

--- 驗證腳本啟動 (V7 - Span Anchor Fix) ---
輸出資料夾: /Volumes/One Touch/result
抽樣數量: 1000 檔案
偵測到 'final_selection_renamed' DataFrame。
已隨機抽取 1000 筆資料進行處理...


處理中... (找到 73 筆 | 檔案錯誤: 0 | 解析錯誤: 0):   1%| | 8/1000 [00:04<04:36,0<?, ?it/s]/var/folders/9b/xrnd2kz13clcbgydjqwnhl7w0000gn/T/ipykernel_24750/1619695573.py:120: XMLParsedAsHTMLWarning: It looks like you're parsing an XML document using an HTML parser. If this really is an HTML document (maybe it's XHTML?), you can ignore or filter this warning. If it's XML, you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the lxml package installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.
  soup = BeautifulSoup(full_text, 'lxml')
處理中... (找到 5465 筆 | 檔案錯誤: 0 | 解析錯誤: 0): 100%|█| 1000/1000 [01:46<0



處理完成，正在寫入總覽報告: /Volumes/One Touch/result/validation_report_8k_files.html
--- 成功！ ---
/Volumes/One Touch/result/validation_report_8k_files.html

V7 腳本診斷報告
總共處理 1000 個檔案。
總共找到 5465 筆關鍵字實例。
找不到檔案: 0 次
關鍵字欄位解析錯誤: 0 次
